# Part 3: App DeploymentExport the best model using TorchScript and run inference.

In [ ]:
import torchimport syssys.path.insert(0, '.')from src.transfer import get_pretrained_modelfrom src.predictor import export_model, load_exported_model, Predictorfrom src.data import get_data_loadersfrom src.helpers import get_device

## 1. Load Best ModelWe use the transfer learning model (ResNet50) as it achieves higher accuracy.

In [ ]:
device = get_device()data_loaders, image_datasets = get_data_loaders("landmark_images", batch_size=1)class_names = image_datasets["train"].classesmodel = get_pretrained_model(num_classes=50)model.load_state_dict(torch.load("transfer_best.pth", map_location=device))model.to(device)model.eval()print("Model loaded successfully")

## 2. Export with TorchScript

In [ ]:
scripted_model = export_model(model, save_path="landmark_model.pt")

## 3. Load Exported Model

In [ ]:
loaded_model = load_exported_model("landmark_model.pt")print("TorchScript model loaded successfully")

## 4. Run InferenceTest the exported model on sample images from the test set.

In [ ]:
import matplotlib.pyplot as pltimport numpy as npfrom torchvision import transformsmean = np.array([0.485, 0.456, 0.406])std = np.array([0.229, 0.224, 0.225])# Get a batch from test setimages, labels = next(iter(data_loaders["test"]))images = images.to(device)with torch.no_grad():    outputs = loaded_model(images[:5])    probs = torch.nn.functional.softmax(outputs, dim=1)    confs, preds = probs.max(1)fig, axes = plt.subplots(1, 5, figsize=(20, 4))for i in range(5):    img = images[i].cpu().numpy().transpose((1, 2, 0))    img = std * img + mean    img = np.clip(img, 0, 1)    axes[i].imshow(img)    true_label = class_names[labels[i]]    pred_label = class_names[preds[i]]    color = "green" if preds[i] == labels[i] else "red"    axes[i].set_title(f"Pred: {pred_label}\nTrue: {true_label}\nConf: {confs[i]:.2f}", color=color, fontsize=9)    axes[i].axis("off")plt.tight_layout()plt.show()

## 5. Generate Submission Package

In [ ]:
import shutilimport ossubmission_dir = "submission"os.makedirs(submission_dir, exist_ok=True)# Copy required filesfor f in ["landmark_model.pt", "cnn_scratch_best.pth", "transfer_best.pth"]:    if os.path.exists(f):        shutil.copy2(f, submission_dir)print("Submission package ready in ./submission/")print("Files:", os.listdir(submission_dir))